# Task 5 — Timing measurements

Measure wall-clock time for the three core operations:

| Operation | Description |
|---|---|
| **Enrollment** | Extract embeddings from all user images → compute averaged vector → write to DB |
| **Verification (1:1)** | Extract probe embedding → compare against one stored user |
| **Identification (1:N)** | Extract probe embedding → HNSW nearest-neighbour search across all enrolled users |

New user images: `data/tttt/Piotr_Zielinski/`

## 1. Setup

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))

from src.model import get_insightface_model, get_embedding
from src.database import EmbeddingDB
from src.authorization import measure_verify_time, measure_identify_time
from src.utils import list_images

warnings.filterwarnings('ignore')

NEW_USER_DIR = Path('../data/tttt')
N_RUNS       = 30    # repetitions for stable timing estimates

print('Loading model...')
app = get_insightface_model('buffalo_l', ctx_id=0)
db  = EmbeddingDB.from_file()

n_enrolled = len(db.get_all_users())
print(f'Users currently in DB : {n_enrolled}')

# Locate new user folder (one subfolder expected)
person_dirs = [d for d in NEW_USER_DIR.iterdir() if d.is_dir()]
assert len(person_dirs) >= 1, f'No subfolders found in {NEW_USER_DIR}'
new_user_dir  = person_dirs[0]
new_user_id   = new_user_dir.name
new_user_imgs = list_images(new_user_dir)
print(f'New user              : {new_user_id}')
print(f'Images available      : {len(new_user_imgs)}')

## 2. Enrollment timing

Measure each phase separately:
- **Embedding extraction** — ArcFace inference per image
- **DB write** — `add_user()` call
- **Total** — full enrollment pipeline (all images → average → store)

In [ ]:
# Load all images for the new user
images_bgr = []
for p in new_user_imgs:
    img = cv2.imread(str(p))
    if img is not None:
        images_bgr.append(img)

print(f'Images loaded : {len(images_bgr)}')

# ── Phase 1: per-image embedding extraction ───────────────────────────────────
embed_times = []
embeddings  = []
for img in images_bgr:
    t0  = time.perf_counter()
    emb = get_embedding(app, img, fallback=True)
    embed_times.append(time.perf_counter() - t0)
    if emb is not None:
        embeddings.append(emb)

t_embed_mean = float(np.mean(embed_times)) * 1000
t_embed_std  = float(np.std(embed_times))  * 1000
print(f'Embedding extraction : {t_embed_mean:.1f} ± {t_embed_std:.1f} ms / image  (n={len(embed_times)})')

# ── Phase 2: averaging + single DB write ──────────────────────────────────────
avg_emb = np.array(embeddings).mean(axis=0)
avg_emb /= np.linalg.norm(avg_emb)

t0 = time.perf_counter()
db.add_user(new_user_id, avg_emb)
t_db_write = (time.perf_counter() - t0) * 1000
print(f'DB write (add_user)  : {t_db_write:.2f} ms')

# ── Phase 3: total enrollment (all images) ────────────────────────────────────
# Remove the user first so we can time the full flow cleanly
db.remove_user(new_user_id)

t0 = time.perf_counter()
embs_all = [get_embedding(app, img, fallback=True) for img in images_bgr]
embs_all = [e for e in embs_all if e is not None]
avg      = np.array(embs_all).mean(axis=0)
avg     /= np.linalg.norm(avg)
db.add_user(new_user_id, avg)
t_total_enroll = (time.perf_counter() - t0) * 1000

print(f'Total enrollment     : {t_total_enroll:.1f} ms  '
      f'({len(images_bgr)} images → 1 vector)')
print(f'Users in DB now      : {len(db.get_all_users())}  (new user added)')

## 3. Verification (1:1) timing

In [ ]:
# Use the first image of the new user as probe
probe_img = images_bgr[0]

verify_times = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    probe_emb = get_embedding(app, probe_img, fallback=True)
    _ = db.query_user(new_user_id, probe_emb)
    verify_times.append((time.perf_counter() - t0) * 1000)

t_verify_mean = float(np.mean(verify_times))
t_verify_std  = float(np.std(verify_times))
t_verify_min  = float(np.min(verify_times))
t_verify_max  = float(np.max(verify_times))

print(f'Verification (1:1) over {N_RUNS} runs:')
print(f'  Mean : {t_verify_mean:.2f} ms')
print(f'  Std  : {t_verify_std:.2f} ms')
print(f'  Min  : {t_verify_min:.2f} ms')
print(f'  Max  : {t_verify_max:.2f} ms')

## 4. Identification (1:N) timing

In [ ]:
identify_times = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    probe_emb = get_embedding(app, probe_img, fallback=True)
    _ = db.query_top1(probe_emb)
    identify_times.append((time.perf_counter() - t0) * 1000)

t_id_mean = float(np.mean(identify_times))
t_id_std  = float(np.std(identify_times))
t_id_min  = float(np.min(identify_times))
t_id_max  = float(np.max(identify_times))

current_db_size = len(db.get_all_users())

print(f'Identification (1:N) over {N_RUNS} runs  [DB size = {current_db_size} users]:')
print(f'  Mean : {t_id_mean:.2f} ms')
print(f'  Std  : {t_id_std:.2f} ms')
print(f'  Min  : {t_id_min:.2f} ms')
print(f'  Max  : {t_id_max:.2f} ms')

## 5. Timing distribution plots

In [ ]:
out_dir = Path('../results/task5')
out_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, times, label, color in [
    (axes[0], verify_times,   f'Verification (1:1)  n={N_RUNS}', 'steelblue'),
    (axes[1], identify_times, f'Identification (1:N)  n={N_RUNS}', 'tomato'),
]:
    ax.hist(times, bins=15, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(np.mean(times), color='k',      ls='--', lw=1.5, label=f'Mean {np.mean(times):.2f} ms')
    ax.axvline(np.median(times), color='gray', ls=':',  lw=1.2, label=f'Median {np.median(times):.2f} ms')
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Count')
    ax.set_title(label)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(out_dir / 'timing_distributions.png', dpi=150)
plt.show()

## 6. 1:N identification time estimate for larger DB sizes

ChromaDB uses an **HNSW index** — query time scales as **O(log N)**, not O(N).  
We estimate by fitting a log curve to the single measured point.

In [ ]:
# Decompose: total time = embedding extraction + DB search
# Embedding time is constant regardless of DB size
t_embed_only_runs = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    _ = get_embedding(app, probe_img, fallback=True)
    t_embed_only_runs.append((time.perf_counter() - t0) * 1000)

t_embed_only = float(np.mean(t_embed_only_runs))
t_db_search  = t_id_mean - t_embed_only   # time spent in HNSW query

print(f'Embedding extraction (constant) : {t_embed_only:.2f} ms')
print(f'DB HNSW search @ {current_db_size} users      : {t_db_search:.2f} ms')

# O(log N) estimate: t_search(N) = t_db_search * log(N) / log(current_db_size)
db_sizes  = [10, 50, 100, 500, 1000, 5000, 10000]
estimated = [
    t_embed_only + t_db_search * np.log(n) / max(np.log(current_db_size), 1e-9)
    for n in db_sizes
]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(db_sizes, estimated, 'o-', color='tomato', lw=2)
ax.axvline(current_db_size, color='k', ls='--', lw=1.2,
           label=f'Current DB ({current_db_size} users, {t_id_mean:.1f} ms)')
ax.axhline(t_embed_only, color='gray', ls=':', lw=1.2,
           label=f'Embedding floor ({t_embed_only:.1f} ms)')
ax.set_xscale('log')
ax.set_xlabel('Number of enrolled users (log scale)')
ax.set_ylabel('Estimated 1:N time (ms)')
ax.set_title('1:N identification time estimate — HNSW O(log N)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_dir / 'identification_scaling.png', dpi=150)
plt.show()

df_est = pd.DataFrame({'DB size': db_sizes, 'Estimated time (ms)': [round(t, 2) for t in estimated]})
display(df_est)

## 7. Cleanup — remove test user from DB

In [ ]:
# Comment this cell out if you want to keep the new user enrolled
if new_user_id in db.get_all_users():
    db.remove_user(new_user_id)
    print(f'Removed "{new_user_id}" from DB.')
print(f'Users in DB : {len(db.get_all_users())}')

## 8. Summary

In [ ]:
print('=' * 55)
print('TASK 5 — TIMING RESULTS')
print('=' * 55)
print(f'  New user              : {new_user_id}')
print(f'  Images used           : {len(images_bgr)}')
print(f'  DB size during test   : {current_db_size} users')
print(f'  Repetitions (N_RUNS)  : {N_RUNS}')
print()
print(f'  Enrollment (full pipeline)    : {t_total_enroll:.1f} ms')
print(f'    per-image embedding         : {t_embed_mean:.1f} ± {t_embed_std:.1f} ms')
print(f'    DB write (add_user)         : {t_db_write:.2f} ms')
print()
print(f'  Verification (1:1) mean       : {t_verify_mean:.2f} ms')
print(f'  Verification (1:1) std        : {t_verify_std:.2f} ms')
print()
print(f'  Identification (1:N) mean     : {t_id_mean:.2f} ms')
print(f'  Identification (1:N) std      : {t_id_std:.2f} ms')
print(f'    of which embedding          : {t_embed_only:.2f} ms  (constant)')
print(f'    of which HNSW search        : {t_db_search:.2f} ms  (O(log N))')
print('=' * 55)